In [ ]:
from pathlib import Path
import pandas as pd

# ============================================================
# TALTECH SEMANTIC PIPELINE
# NOTEBOOK 02 — IFC / UUID RECONCILIATION
# ============================================================

# ============================================================
# 1. PROJECT PATHS
# ============================================================

PROJECT_DIR = Path.cwd()

DATA_DIR = PROJECT_DIR / "data"

OUTPUT_DIR = PROJECT_DIR / "outputs"

EXTRACTION_DIR = (
    OUTPUT_DIR /
    "01_extraction"
)

OUT = (
    OUTPUT_DIR /
    "02_reconciliation"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)

# Input files
IFC_FILE = (
    DATA_DIR /
    "DS3_TalTech_V4.ifc"
)

DEVICES_FILE = (
    EXTRACTION_DIR /
    "01_selected_devices.csv"
)

print("Project directory :", PROJECT_DIR)
print("IFC model         :", IFC_FILE)
print("Selected devices  :", DEVICES_FILE)
print("Output directory  :", OUT)

# ============================================================
# 2. INPUT VERIFICATION
# ============================================================

assert IFC_FILE.exists(), (
    "IFC file not found:\n"
    f"{IFC_FILE}"
)

assert DEVICES_FILE.exists(), (
    "Selected devices file not found.\n"
    "Run Notebook 01 first:\n"
    f"{DEVICES_FILE}"
)

print("\n✓ IFC model found")
print("✓ Selected devices file found")

# ============================================================
# 3. LOAD SELECTED OPERATIONAL UUIDs
# ============================================================

devices_df = pd.read_csv(
    DEVICES_FILE,
    dtype=str
)

assert "device_id" in devices_df.columns, (
    "Column 'device_id' not found in "
    "01_selected_devices.csv"
)

devices = (
    devices_df["device_id"]
    .dropna()
    .astype(str)
    .str.strip()
    .tolist()
)

print("\n====================================")
print("SELECTED OPERATIONAL DEVICES")
print("====================================")

print(
    "Number of selected devices:",
    len(devices)
)

for uuid in devices:
    print(" -", uuid)

# ============================================================
# 4. READ IFC AS TEXT
# ============================================================

ifc_text = IFC_FILE.read_text(
    encoding="utf-8",
    errors="ignore"
)

print("\nIFC file loaded.")
print(
    "IFC text size:",
    len(ifc_text),
    "characters"
)

# ============================================================
# 5. SEARCH EXACT UUID CORRESPONDENCE IN IFC
# ============================================================

uuid_checks = []

for uuid in devices:

    occurrences = (
        ifc_text
        .lower()
        .count(
            uuid.lower()
        )
    )

    uuid_checks.append(
        {
            "operational_uuid":
                uuid,

            "exact_uuid_occurrences_in_ifc":
                occurrences,

            "native_direct_match":
                occurrences > 0
        }
    )

uuid_checks = pd.DataFrame(
    uuid_checks
)

print("\n====================================")
print("UUID → IFC DIRECT MATCH")
print("====================================")

print(
    uuid_checks.to_string(
        index=False
    )
)

DIRECT_MATCH_FILE = (
    OUT /
    "01_uuid_ifc_direct_match.csv"
)

uuid_checks.to_csv(
    DIRECT_MATCH_FILE,
    index=False
)

# ============================================================
# 6. IFC ENTITY INVENTORY
# ============================================================

# We first try to use IfcOpenShell.
# If it is unavailable, the UUID reconciliation table
# will still be created.

inventory_rows = []
spaces_df = pd.DataFrame()

try:

    import ifcopenshell

    print(
        "\n✓ IfcOpenShell available"
    )

    ifc = ifcopenshell.open(
        str(
            IFC_FILE
        )
    )

    entity_types = [
        "IfcSpace",
        "IfcSensor",
        "IfcFlowMeter",
        "IfcDistributionSystem",
        "IfcAirTerminal",
        "IfcUnitaryEquipment",
        "IfcAirToAirHeatRecovery"
    ]

    for entity_type in entity_types:

        try:

            count = len(
                ifc.by_type(
                    entity_type
                )
            )

        except Exception:

            count = None

        inventory_rows.append(
            {
                "entity_type":
                    entity_type,

                "count":
                    count
            }
        )

    inventory_df = pd.DataFrame(
        inventory_rows
    )

    inventory_df.to_csv(
        OUT /
        "02_ifc_entity_inventory.csv",
        index=False
    )

    print("\n====================================")
    print("IFC ENTITY INVENTORY")
    print("====================================")

    print(
        inventory_df.to_string(
            index=False
        )
    )

    # ========================================================
    # 7. EXTRACT IFC SPACES
    # ========================================================

    spaces = []

    for space in ifc.by_type(
        "IfcSpace"
    ):

        spaces.append(
            {
                "ifc_global_id":
                    getattr(
                        space,
                        "GlobalId",
                        ""
                    ),

                "ifc_name":
                    getattr(
                        space,
                        "Name",
                        ""
                    ),

                "ifc_long_name":
                    getattr(
                        space,
                        "LongName",
                        ""
                    ),

                "ifc_object_type":
                    getattr(
                        space,
                        "ObjectType",
                        ""
                    )
            }
        )

    spaces_df = pd.DataFrame(
        spaces
    )

    spaces_df.to_csv(
        OUT /
        "03_ifc_spaces.csv",
        index=False
    )

    print(
        "\nIfcSpace entities:",
        len(spaces_df)
    )

    if len(spaces_df) > 0:

        print(
            "\nFirst IFC spaces:"
        )

        print(
            spaces_df.head(
                10
            ).to_string(
                index=False
            )
        )

except ImportError:

    print(
        "\nWARNING:"
        " IfcOpenShell is not installed."
    )

    print(
        "The reconciliation table will still "
        "be created, but IFC entity extraction "
        "will be skipped."
    )

# ============================================================
# 8. CREATE FORMAL RECONCILIATION TABLE
# ============================================================

reconciliation = (
    uuid_checks.copy()
)

reconciliation[
    "ifc_global_id"
] = ""

reconciliation[
    "ifc_space_name"
] = ""

reconciliation[
    "mapping_origin"
] = "none"

reconciliation[
    "mapping_status"
] = reconciliation[
    "native_direct_match"
].map(
    {
        True:
            "NATIVE_MATCH_REVIEW_REQUIRED",

        False:
            "PENDING_ENRICHMENT"
    }
)

# IMPORTANT:
# No factual physical location is asserted
# unless supported by source evidence.

reconciliation[
    "factual_location_claim"
] = False

reconciliation[
    "mapping_confidence"
] = ""

reconciliation[
    "provenance"
] = (
    "TalTech operational CSV UUID; "
    "TalTech IFC inspected for "
    "machine-readable correspondence"
)

reconciliation[
    "approved_for_rdf"
] = False

reconciliation[
    "review_note"
] = (
    "No physical UUID-to-IfcSpace mapping "
    "is asserted automatically. "
    "Experimental semantic enrichment "
    "must be reviewed before RDF integration."
)

# ============================================================
# 9. SAVE RECONCILIATION TABLE
# ============================================================

RECON_FILE = (
    OUT /
    "04_reconciliation_review_table.csv"
)

reconciliation.to_csv(
    RECON_FILE,
    index=False
)

print("\n====================================")
print("RECONCILIATION TABLE")
print("====================================")

print(
    reconciliation.to_string(
        index=False
    )
)

# ============================================================
# 10. SUMMARY
# ============================================================

native_matches = int(
    reconciliation[
        "native_direct_match"
    ].sum()
)

pending = int(
    (
        reconciliation[
            "mapping_status"
        ]
        ==
        "PENDING_ENRICHMENT"
    ).sum()
)

print("\n====================================")
print("RECONCILIATION SUMMARY")
print("====================================")

print(
    "Operational UUIDs:",
    len(
        reconciliation
    )
)

print(
    "Native UUID→IFC matches:",
    native_matches
)

print(
    "Pending enrichment:",
    pending
)

print(
    "Approved mappings:",
    int(
        reconciliation[
            "approved_for_rdf"
        ].astype(
            str
        ).str.lower().isin(
            [
                "true",
                "1",
                "yes"
            ]
        ).sum()
    )
)

print(
    "\nReconciliation file created:"
)

print(
    RECON_FILE
)

# ============================================================
# 11. OUTPUT FILES
# ============================================================

print("\n====================================")
print("FILES CREATED")
print("====================================")

for output_file in sorted(
    OUT.iterdir()
):

    print(
        " -",
        output_file.name
    )